# Homework: Can a GAN Beat SMOTE for Imbalanced Text?

**Course:** SEAS 8525, Computer Vision and Generative AI

In class we saw that a GAN cannot easily generate *words*: words are discrete, so the discriminator's gradient cannot flow back into the generator. In this homework we sidestep that problem the same way SMOTE does. We turn each text message into a **sentence embedding** (384 numbers) and run the GAN in that continuous space.

**Your question:** with very little minority-class data, does GAN oversampling help a spam classifier more than SMOTE does? And is the GAN data actually realistic, varied, and not copied?

## What you will do
1. Load the SMS Spam Collection and keep only a **small number of spam messages** for training.
2. Embed every message with a small sentence-embedding model.
3. Balance the training set three ways: **nothing**, **SMOTE**, and **a GAN** trained only on spam embeddings.
4. Train the same classifier on each version and test on the **same real test set**.
5. Check the GAN data four ways: **usefulness**, **realism** (Isolation Forest, One-Class SVM), **a classifier two-sample test**, and **variety and copying**.
6. Look at what the GAN "wrote" by finding the nearest real message to its samples.
7. Compare with a **WGAN-GP** version of the GAN (already built in).

**You do not need to write any code.** Set your ID in the first code cell, then run every cell in order (Runtime > Run all works). For report section 6, change values in the **Experiment settings** cell and run again.

## Your personal settings
Your random seed and your number of spam training messages come from the **last two digits of your student ID** (set it in the first code cell). Your numbers will differ from your classmates', so your report must be based on **your own results**.

## What to hand in
1. **This notebook**, with all cells run (File > Download > .ipynb).
2. **A report** (PDF, 2 to 3 pages) with these sections:
   1. **Setup:** your seed, your spam count, your GAN design (layer sizes, learning rates, steps), and anything you changed.
   2. **Training behavior:** your G and D loss curves. Do they settle or oscillate? Did D ever win completely? *Your losses do not tell you whether the samples are good. What did you use instead?*
   3. **Results table:** none vs SMOTE vs GAN, with minority F1, recall, precision, Isolation Forest and One-Class SVM acceptance, two-sample AUC, diversity, and memorization distance. Include the reference rows (held-out real spam, real ham, and random noise).
   4. **Interpretation**, using your own numbers:
      1. Which method gave the best minority F1? Was the difference meaningful, or within the spread across seeds?
      2. Did Isolation Forest, One-Class SVM, and the two-sample test agree? If not, what does that tell you about each check?
      3. Did your GAN show mode collapse? Which number shows it?
      4. Is the GAN copying the training data? How do you know?
      5. Show 3 "nearest real message" examples. What kind of spam did the GAN seem to produce, and what did it miss?
   5. **When would you NOT use a GAN here?** Based on your results, when is SMOTE the better choice?
   6. **One change you tried:** change one value in the Experiment settings cell (for example the GAN steps, learning rate, or noise size), rerun, and report before and after numbers. Also compare the GAN with WGAN-GP (section 10). A change that did not help, explained well, is fine.
   7. **Research link (one paragraph):** could this approach help in your own research topic? Why or why not?

Follow the course policy on AI tools, and state in one line how you used them, if at all.

**Runtime:** a free Colab CPU runtime is enough (about 5 to 10 minutes in total). A GPU runtime makes the GAN steps faster.


## 1. Setup

In [ ]:
# Install the two libraries Colab may not have
!pip -q install sentence-transformers imbalanced-learn

In [ ]:
# Put the LAST TWO DIGITS of your student ID here (0 to 99)
STUDENT_ID_LAST2 = 99   # for example 7 or 42

assert STUDENT_ID_LAST2 is not None, "Set STUDENT_ID_LAST2 first."
SEED = 1000 + STUDENT_ID_LAST2          # your random seed
N_SPAM_TRAIN = 50 + STUDENT_ID_LAST2    # your spam training messages (50 to 149)
print(f"SEED = {SEED},  N_SPAM_TRAIN = {N_SPAM_TRAIN}")

In [ ]:
# ---- Experiment settings (change these for report section 6) ----
GAN_STEPS = 3000      # training steps for the GAN and WGAN-GP
GAN_LR = 2e-4         # learning rate for the GAN
Z_DIM = 64            # size of the noise vector z
HIDDEN = 256          # width of the generator's hidden layers
PCA_DIM = 32          # size of the reduced space used by the outlier detectors


In [ ]:
import io, random, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load the data
The SMS Spam Collection: about 5,500 text messages, labeled ham (0) or spam (1). We try the UCI archive first and fall back to Hugging Face.

In [ ]:
def load_sms_spam():
    url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
    try:
        raw = urllib.request.urlopen(url, timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as zf:
            text = zf.read("SMSSpamCollection").decode("utf-8")
        rows = [line.split("\t", 1) for line in text.strip().split("\n")]
        df = pd.DataFrame(rows, columns=["label", "text"])
        df["label"] = (df["label"] == "spam").astype(int)
        print("Loaded from UCI")
        return df
    except Exception as e:
        print("UCI download failed, trying Hugging Face:", e)
        from datasets import load_dataset
        ds = load_dataset("ucirvine/sms_spam", split="train")
        df = pd.DataFrame({"text": ds["sms"], "label": ds["label"]})  # 1 = spam
        print("Loaded from Hugging Face")
        return df

df = load_sms_spam()
print(df.shape)
print(df["label"].value_counts().rename({0: "ham", 1: "spam"}))
df.head()

## 3. Make it hard: keep only a few spam messages for training
We split into train and test first. The **test set stays untouched and fully real**. Then we shrink the spam part of the training set to your `N_SPAM_TRAIN`.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)

spam_train = train_df[train_df["label"] == 1].sample(n=N_SPAM_TRAIN, random_state=SEED)
ham_train = train_df[train_df["label"] == 0]
train_small = pd.concat([ham_train, spam_train]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training set:", train_small["label"].value_counts().rename({0: "ham", 1: "spam"}).to_dict())
print("Test set:    ", test_df["label"].value_counts().rename({0: "ham", 1: "spam"}).to_dict())

## 4. Embed every message
`all-MiniLM-L6-v2` turns each message into 384 numbers. Embeddings are normalized to length 1, so every message is a point on a sphere.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2", device=str(device))

def embed(texts):
    return encoder.encode(list(texts), batch_size=128, normalize_embeddings=True,
                          show_progress_bar=True).astype(np.float32)

X_train = embed(train_small["text"]); y_train = train_small["label"].values
X_test = embed(test_df["text"]);      y_test = test_df["label"].values

X_spam_train = X_train[y_train == 1]       # what the GAN will learn from
X_spam_test = X_test[y_test == 1]          # held-out REAL spam, used only for checking
N_NEEDED = int((y_train == 0).sum() - (y_train == 1).sum())   # samples needed to balance
DIM = X_train.shape[1]
print("Embedding size:", DIM, "| spam to generate:", N_NEEDED)

## 5. Baseline and SMOTE
Same classifier every time: logistic regression. We report precision, recall, and F1 **for the spam class** on the real test set.

In [ ]:
def unit(X):
    """Rescale rows to length 1, like the real embeddings."""
    return X / np.linalg.norm(X, axis=1, keepdims=True).clip(min=1e-12)

def evaluate(X_tr, y_tr, name):
    clf = LogisticRegression(max_iter=3000)
    clf.fit(X_tr, y_tr)
    pred = clf.predict(X_test)
    return {"method": name,
            "F1 (spam)": f1_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred)}

results = [evaluate(X_train, y_train, "none")]

from imblearn.over_sampling import SMOTE
X_res, y_res = SMOTE(random_state=SEED, k_neighbors=5).fit_resample(X_train, y_train)
X_smote_new = unit(X_res[len(X_train):])          # SMOTE appends its new samples at the end
X_tr_smote = np.vstack([X_train, X_smote_new])
y_tr_smote = np.concatenate([y_train, np.ones(len(X_smote_new), dtype=int)])
results.append(evaluate(X_tr_smote, y_tr_smote, "SMOTE"))

pd.DataFrame(results).round(3)

> **How to interpret your results: baseline vs. SMOTE**
>
> Both methods use the **same logistic-regression classifier** and are evaluated on the **same untouched real test set**. **Recall** is the fraction of actual spam detected; **precision** is the fraction of predicted spam that truly is spam; and **F1** summarizes the balance between precision and recall.
>
> Compare **your own** baseline and SMOTE rows. Did recall increase? Did precision decrease? Did F1 improve overall? More synthetic training examples may help the classifier detect spam, but improvements are **not guaranteed**. Even if SMOTE improves F1, that does **not** prove its synthetic embeddings are realistic. Report the values from **your run** and explain any trade-off.

## 6. The GAN
A small GAN in embedding space, trained **only on your spam embeddings**.

* **Generator:** noise `z` (64 numbers) → 256 → 256 → 384, LeakyReLU(0.2) between layers. Its output is rescaled to length 1 (`F.normalize`), so fakes live on the same sphere as real embeddings.
* **Discriminator:** 384 → 256 → 128 → 1 score (a logit, **no sigmoid**: `BCEWithLogitsLoss` adds it). LeakyReLU(0.2), plus Dropout(0.3) after the first layer because we have very little data.

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM, out_dim=DIM, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, out_dim))

    def forward(self, z):
        return F.normalize(self.net(z), dim=1)

class Discriminator(nn.Module):
    def __init__(self, in_dim=DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)      # one score (logit) per sample

G_test, D_test = Generator(), Discriminator()
z = torch.randn(8, Z_DIM)
assert G_test(z).shape == (8, DIM)
assert D_test(G_test(z)).shape == (8,)
print("Shapes OK")

### Training loop: the same take-turns game from class
1. **Update D:** real spam → label 1, fakes → label 0 (BCE).
2. **Update G:** the **non-saturating loss**, BCE with the fakes labeled **1**. D is not updated in this step.

In [ ]:
def train_gan(X_real, steps=GAN_STEPS, batch=64, lr=GAN_LR, seed=SEED, log_every=10):
    set_seed(seed)
    G, D = Generator().to(device), Discriminator().to(device)
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    real_all = torch.tensor(X_real, device=device)
    hist = {"step": [], "D": [], "G": []}

    for step in range(steps):
        idx = torch.randint(0, len(real_all), (batch,), device=device)
        real = real_all[idx]
        ones = torch.ones(batch, device=device)
        zeros = torch.zeros(batch, device=device)

        # ---- 1. Update D ----
        fake = G(torch.randn(batch, Z_DIM, device=device))
        loss_D = bce(D(real), ones) + bce(D(fake.detach()), zeros)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # ---- 2. Update G (non-saturating) ----
        fake = G(torch.randn(batch, Z_DIM, device=device))
        loss_G = bce(D(fake), ones)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        if step % log_every == 0:
            hist["step"].append(step); hist["D"].append(loss_D.item()); hist["G"].append(loss_G.item())
    return G, hist

In [ ]:
G, hist = train_gan(X_spam_train)

plt.figure(figsize=(9, 4))
plt.plot(hist["step"], hist["D"], label="D loss")
plt.plot(hist["step"], hist["G"], label="G loss")
plt.axhline(2 * np.log(2), ls="--", c="gray", lw=1, label="D at a coin flip (1.39)")
plt.axhline(np.log(2), ls=":", c="gray", lw=1, label="G at a coin flip (0.69)")
plt.xlabel("training step"); plt.ylabel("loss"); plt.legend(); plt.title("GAN training")
plt.show()

> **How to read the GAN training losses (use your own run)**
>
> This notebook plots two **binary cross-entropy (BCE)** losses. **D loss** is the **sum** of BCE on real spam (target 1) **plus** BCE on generated spam (target 0). **G loss** is **non-saturating BCE** on generated spam with target 1 (G wants D to call it real). **Lower is better for the network being updated**, but the networks compete, so the curves need not both decrease or converge smoothly.
>
> **Reference values, not required targets:** If D predicts **0.5** for both real and fake samples, its summed loss is `-log(0.5)-log(0.5) ≈ 1.39`; G's loss at `D(fake)=0.5` is `-log(0.5) ≈ 0.69`. These are the dotted lines in the plot. The exact theoretical equilibrium requires the **real and generated distributions to match** and an **optimal D**. **D loss near 1.39 does not, by itself, prove equilibrium**: an untrained D could also guess randomly.
>
> **Common patterns:** **D loss near 0** with **large G loss** suggests D easily rejects fakes and G is struggling. **D loss around 1.39** and **G loss around 0.69** may indicate balance, but only if the other quality checks agree. **Oscillation or fluctuations** are common because G and D take alternating updates; neither loss must decrease steadily. If **both losses become NaN or Inf**, that signals numerical instability rather than success. Values **above these references can occur**; these losses do not have a universal fixed upper bound. The specific numbers depend on the seed, settings, and training state.
>
> **What to report:** Describe the **pattern over training steps**, whether D seemed to dominate or both networks traded improvements, and whether the **real-test F1 and realism/diversity checks** support what the curves seem to suggest. Do **not** rank GAN versus SMOTE by training loss; SMOTE has no adversarial loss.

In [ ]:
def sample_gan(G, n):
    G.eval()
    with torch.no_grad():
        X = G(torch.randn(n, Z_DIM, device=device)).cpu().numpy()
    G.train()
    return X

X_gan_new = sample_gan(G, N_NEEDED)
X_tr_gan = np.vstack([X_train, X_gan_new])
y_tr_gan = np.concatenate([y_train, np.ones(N_NEEDED, dtype=int)])
results.append(evaluate(X_tr_gan, y_tr_gan, "GAN"))

utility = pd.DataFrame(results).set_index("method").round(3)
utility

> **How to interpret your results: GAN oversampling**
>
> Compare **no oversampling, SMOTE, and GAN** on the **same untouched real test set**. Higher spam-class **F1** indicates a better precision/recall balance; **recall** measures how many actual spam messages are found, and **precision** measures how many predicted spam messages really are spam. A method can increase recall while lowering precision. There is **no required F1 score or guaranteed winner**: use the numbers from your own run. GAN loss curves alone do not establish whether its generated data are useful.

## 7. Is the GAN data any good? Four checks
We compare the GAN samples against two **reference** groups:
* **held-out real spam** (`X_spam_test`): the best we could hope for,
* **real ham**: real text, but the wrong class. A good check should reject it,
* **random noise** on the sphere: clearly fake.

Without these two reference points, the numbers are hard to read.

In [ ]:
rng = np.random.default_rng(SEED)
X_noise = unit(rng.normal(size=(len(X_spam_test), DIM)).astype(np.float32))

X_ham_test = X_test[y_test == 0]
X_ham_ref = X_ham_test[rng.choice(len(X_ham_test), len(X_spam_test), replace=False)]

groups = {"real spam (held out)": X_spam_test,
          "GAN": X_gan_new,
          "SMOTE": X_smote_new,
          "real ham (wrong class)": X_ham_ref,
          "random noise": X_noise}

### Check 3: realism with Isolation Forest and One-Class SVM
Both are fit on your **real training spam**. We report the share of each group they accept as "normal".

Read the reference rows first: **held-out real spam** should be mostly accepted, **ham** and **noise** mostly rejected. If not, the detector itself is unreliable here (try another `PCA_DIM` in the settings cell), and that is worth discussing in your report.

⚠️ These detectors only check that a sample falls **inside** the real data region. A GAN that makes nearly the same sample again and again can still score very well. That is why checks 4 and 5 matter.

In [ ]:
from sklearn.decomposition import PCA

# Outlier detectors struggle in 384 dimensions with only ~100 training points,
# so we fit them in a smaller space (PCA_DIM, from the settings cell) found by PCA on all real training embeddings.
pca = PCA(n_components=PCA_DIM, random_state=SEED).fit(X_train)
P = pca.transform

iso = IsolationForest(n_estimators=300, random_state=SEED).fit(P(X_spam_train))
ocsvm = OneClassSVM(nu=0.1, gamma="scale").fit(P(X_spam_train))

realism = pd.DataFrame({
    "IsoForest accepts": {k: float((iso.predict(P(v)) == 1).mean()) for k, v in groups.items()},
    "OC-SVM accepts": {k: float((ocsvm.predict(P(v)) == 1).mean()) for k, v in groups.items()},
})
realism.round(3)

> **How to interpret your results: acceptance rates**
>
> Each acceptance rate is the **fraction a detector marks as normal**, between **0 and 1** (multiply by 100 for a percentage), **not classification accuracy**. Start with the reference groups: ideally **held-out real spam** has relatively high acceptance, while **real ham and random noise** have lower acceptance. If noise or ham is accepted at a high rate, question the detector's usefulness under these settings. Next compare GAN and SMOTE with those references. **High GAN acceptance alone does not prove realistic or diverse samples**; a detector can accept repetitive or unrealistic points. Different detectors may disagree, and that disagreement is an important finding.

### Check 4: can a classifier tell real from synthetic? (classifier two-sample test)
Train a simple classifier to separate **held-out real spam** from each group. AUC near **0.5** means it cannot tell them apart (good). AUC near **1.0** means they are easy to separate.

This is exactly a discriminator, trained after the fact.

In [ ]:
def two_sample_auc(A, B, seed=SEED):
    """5-fold cross-validated AUC of a classifier separating A (label 0) from B (label 1)."""
    n = min(len(A), len(B))
    r = np.random.default_rng(seed)
    A = A[r.choice(len(A), n, replace=False)]
    B = B[r.choice(len(B), n, replace=False)]
    X = np.vstack([A, B])
    y = np.concatenate([np.zeros(n), np.ones(n)])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    return float(cross_val_score(LogisticRegression(max_iter=3000), X, y, cv=cv, scoring="roc_auc").mean())

# Sanity check: real training spam vs held-out real spam should give roughly 0.5
c2st = {"real spam (held out)": two_sample_auc(X_spam_test, X_spam_train)}
for k in ["GAN", "SMOTE", "real ham (wrong class)", "random noise"]:
    c2st[k] = two_sample_auc(X_spam_test, groups[k])
pd.Series(c2st, name="two-sample AUC").round(3)

> **How to interpret your results: two-sample AUC**
>
> This classifier tries to distinguish **held-out real spam** from each comparison group, **not spam from ham**. **AUC near 0.5** means little detectable separation in this test; **near 1.0** means easy separation, which is **unfavorable for synthetic-data realism**. Values below 0.5 may mean the score direction is reversed or reflect sampling variability; do not automatically call them better. Compare the GAN and SMOTE with the **held-out-real-spam reference**. A strong outlier-detector acceptance rate can coexist with a high two-sample AUC: the tests ask different questions. No fixed AUC is guaranteed.

### Check 5: variety and copying
* **Diversity:** average distance between pairs of samples in a group. Much smaller than real spam = **mode collapse**.
* **Memorization:** median distance from each sample to its **nearest real training spam**. Much smaller than for held-out real spam = the generator is **copying**.

In [ ]:
def diversity(X, max_n=500, seed=SEED):
    r = np.random.default_rng(seed)
    if len(X) > max_n:
        X = X[r.choice(len(X), max_n, replace=False)]
    D = pairwise_distances(X)
    return float(D[np.triu_indices(len(X), k=1)].mean())

nn_train = NearestNeighbors(n_neighbors=1).fit(X_spam_train)
def memorization(X):
    d, _ = nn_train.kneighbors(X)
    return float(np.median(d))

variety = pd.DataFrame({
    "diversity": {k: diversity(v) for k, v in groups.items()},
    "distance to nearest train spam": {k: memorization(v) for k, v in groups.items()},
})
variety.round(3)

> **How to interpret your results: distances and diversity**
>
> Distances are **Euclidean distances between normalized 384-dimensional embeddings**, not word counts. For unit-length vectors, `distance = sqrt(2 - 2 × cosine_similarity)`; distance ranges from **0** (identical vectors) to **2** (opposite vectors), with **1.414** corresponding to cosine similarity **0**. Smaller means closer **in embedding space**, not necessarily identical text.
>
> **Diversity** is the average distance between pairs within a group. Compare GAN and SMOTE against **held-out real spam**: much lower GAN diversity may indicate limited variety or possible mode collapse; a similar value is more encouraging, but not proof of quality. **Nearest-training-spam distance** is the median distance from each synthetic sample to the closest *real training spam* embedding. Compare with held-out real spam: unusually tiny distances may suggest memorization, but do **not** prove verbatim copying. SMOTE can naturally be close because it interpolates between training points. **Do not use an absolute distance cutoff** without a reference distribution.

### All checks in one table (copy this into your report)

In [ ]:
checks = realism.join(pd.Series(c2st, name="two-sample AUC")).join(variety)
summary = checks.join(utility, how="outer")
summary = summary.reindex(["none", "SMOTE", "GAN", "real spam (held out)", "real ham (wrong class)", "random noise"])
summary.round(3)

> **How to interpret your results: combined table**
>
> Use spam-class **F1/precision/recall** for downstream classification usefulness; **Isolation Forest and One-Class SVM acceptance** for coarse in-distribution checks; **two-sample AUC** for distinguishability; and **diversity/nearest-training distance** for variety and possible memorization. **Compare with the reference rows first**, then compare GAN, SMOTE, and WGAN-GP where available. These measures can disagree, so explain why instead of picking the most favorable number. Your run may produce a different ranking from anyone else's.

## 8. What did the GAN "write"?
A GAN sample is a point in embedding space, not a sentence. To get a feel for it, we find the **nearest real training message** and print it.

In [ ]:
nn_all = NearestNeighbors(n_neighbors=1).fit(X_train)
d, idx = nn_all.kneighbors(X_gan_new[:8])
for i, (dist, j) in enumerate(zip(d[:, 0], idx[:, 0])):
    row = train_small.iloc[j]
    tag = "SPAM" if row["label"] == 1 else "ham "
    print(f"GAN sample {i+1}: nearest = [{tag}] (distance {dist:.3f})  {row['text'][:110]}")

> **How to interpret your output**
>
> **Interpreting these examples:** The printed messages are **nearest real neighbors**, not sentences written by the GAN. The distance measures embedding similarity: smaller values mean a generated point is closer to that training message. Reading a nearby spam message can suggest a theme, but you **cannot infer the exact text** of a generated embedding. Also, this display searches **all training messages**, so the nearest neighbor can be ham or spam; the memorization metric above searches **training spam only**.

## 9. Is the difference real? Repeat over 3 seeds
One run can be lucky. This reruns SMOTE and the GAN with three seeds on the **same** data split and reports mean ± spread of the spam F1. (Takes a few minutes on CPU.)

In [ ]:
def f1_for_seed(s):
    Xr, yr = SMOTE(random_state=s, k_neighbors=5).fit_resample(X_train, y_train)
    Xs = unit(Xr[len(X_train):])
    f_smote = evaluate(np.vstack([X_train, Xs]), np.concatenate([y_train, np.ones(len(Xs), dtype=int)]), "SMOTE")["F1 (spam)"]
    Gs, _ = train_gan(X_spam_train, seed=s)
    Xg = sample_gan(Gs, N_NEEDED)
    f_gan = evaluate(np.vstack([X_train, Xg]), np.concatenate([y_train, np.ones(N_NEEDED, dtype=int)]), "GAN")["F1 (spam)"]
    return f_smote, f_gan

runs = np.array([f1_for_seed(SEED + k) for k in range(3)])
f_none = utility.loc["none", "F1 (spam)"]
print(f"none : {f_none:.3f}  (no randomness in this one)")
print(f"SMOTE: {runs[:, 0].mean():.3f} ± {runs[:, 0].std():.3f}")
print(f"GAN  : {runs[:, 1].mean():.3f} ± {runs[:, 1].std():.3f}")

> **How to interpret your results: repeated seeds**
>
> Report the **mean and spread** of F1 over the three runs. A larger spread means results are less stable across these training seeds; a small difference between methods relative to the observed spread is less convincing than a large, consistent gap. These are repeated runs on the **same fixed train/test split**, not independent test datasets. Three seeds are useful for an initial stability check, but not enough for a broad statistical conclusion. Use **your own computed means and spread**; there is no expected fixed result.

## 10. WGAN-GP
The same job done by a **WGAN-GP** from class:
* the critic is the same network as D, with **no sigmoid and no BCE**,
* critic loss = avg f(fake) − avg f(real) + λ · avg(‖∇f(x̂)‖ − 1)², with x̂ random points between real and fake, λ = 10,
* **5 critic steps** per G step, Adam with lr = 1e-4 and betas = (0.0, 0.9),
* G loss = − avg f(G(z)).

The cell trains it and prints the same numbers, so you can compare it with the GAN above.

In [ ]:
def train_wgan_gp(X_real, steps=GAN_STEPS, batch=64, n_critic=5, lam=10.0, lr=1e-4, seed=SEED, log_every=10):
    set_seed(seed)
    G = Generator().to(device)
    C = nn.Sequential(nn.Linear(DIM, 256), nn.LeakyReLU(0.2),
                      nn.Linear(256, 128), nn.LeakyReLU(0.2),
                      nn.Linear(128, 1)).to(device)          # critic: no sigmoid, no dropout
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.0, 0.9))
    opt_C = torch.optim.Adam(C.parameters(), lr=lr, betas=(0.0, 0.9))
    real_all = torch.tensor(X_real, device=device)
    hist = {"step": [], "critic": [], "G": []}

    for step in range(steps):
        for _ in range(n_critic):
            real = real_all[torch.randint(0, len(real_all), (batch,), device=device)]
            fake = G(torch.randn(batch, Z_DIM, device=device)).detach()
            eps = torch.rand(batch, 1, device=device)
            x_hat = (eps * real + (1 - eps) * fake).requires_grad_(True)
            grad = torch.autograd.grad(C(x_hat).sum(), x_hat, create_graph=True)[0]
            gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()
            loss_C = C(fake).mean() - C(real).mean() + lam * gp
            opt_C.zero_grad(); loss_C.backward(); opt_C.step()

        loss_G = -C(G(torch.randn(batch, Z_DIM, device=device))).mean()
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        if step % log_every == 0:
            hist["step"].append(step); hist["critic"].append(loss_C.item()); hist["G"].append(loss_G.item())
    return G, hist

G_w, hist_w = train_wgan_gp(X_spam_train)
plt.figure(figsize=(9, 4))
plt.plot(hist_w["step"], [-c for c in hist_w["critic"]], label="-(critic loss) = estimate of W (incl. penalty)")
plt.xlabel("training step"); plt.legend(); plt.title("WGAN-GP training"); plt.show()

X_wgan_new = sample_gan(G_w, N_NEEDED)
res_w = evaluate(np.vstack([X_train, X_wgan_new]), np.concatenate([y_train, np.ones(N_NEEDED, dtype=int)]), "WGAN-GP")
print(res_w)
print("IsoForest accepts:", (iso.predict(P(X_wgan_new)) == 1).mean().round(3),
      "| two-sample AUC:", round(two_sample_auc(X_spam_test, X_wgan_new), 3),
      "| diversity:", round(diversity(X_wgan_new), 3),
      "| distance to train:", round(memorization(X_wgan_new), 3))

> **How to interpret your results: WGAN-GP**
>
> Compare WGAN-GP's **spam-class F1, precision/recall, detector acceptance, two-sample AUC, diversity, and nearest-training distance** with the standard GAN and SMOTE. A higher F1 is better for the downstream classifier, but it does not prove the samples are more realistic. **Its critic loss is not binary cross-entropy**; it may be negative, and its magnitude is not directly comparable with the standard GAN's D or G losses. The displayed `-(critic loss)` curve also includes the **gradient-penalty term**, so it is **not a pure Wasserstein-distance estimate**. Look for training stability and compare quality metrics instead of expecting a particular curve value.

## Done
Download this notebook (File > Download > .ipynb) and write your report using the sections listed at the top.